# Setup & Imports

In [1]:
import os
import json
import string
import time
from bioblend.toolshed import ToolShedInstance

# Konfiguration
TOOLSHED_URL = "https://toolshed.g2.bx.psu.edu"  # oder https://toolshed.eu
SAVE_PATH = "toolshed_tools_partial.json"
PROGRESS_PATH = "progress.log"
PAGE_SIZE = 25
SLEEP_BETWEEN_PAGES = 0.5

# Verbindung
ts = ToolShedInstance(url=TOOLSHED_URL)
tstc = ts.tools


# Laden vorhandener Daten & Fortschritt

In [2]:
# Vorhandene Daten laden
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, "r", encoding="utf-8") as f:
        all_tools = json.load(f)
    seen_ids = {h["tool"]["id"] for h in all_tools if "tool" in h}
    print(f"[INFO] Wiederaufnahme: {len(all_tools)} Tools geladen.")
else:
    all_tools, seen_ids = [], set()

# Zweistellige Buchstaben-Kombis + einzelne Ziffern
letters = string.ascii_lowercase
digits = string.digits
queries = [a + b for a in letters for b in letters] + list(digits)

# Fortschritt prüfen
if os.path.exists(PROGRESS_PATH):
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        last_done = f.read().strip()
    if last_done in queries:
        start_index = queries.index(last_done) + 1
        print(f"[INFO] Fortsetze nach '{last_done}' → starte bei '{queries[start_index]}'")
    else:
        start_index = 0
else:
    start_index = 0

# Hauptschleife (resumable)

In [3]:
# Hauptschleife
for i, q in enumerate(queries[start_index:], start=start_index + 1):
    print(f"\n[{i:03d}/{len(queries)}] Suche nach '{q}'...")
    page = 1
    while True:
        try:
            res = tstc.search_tools(q=q, page=page, page_size=PAGE_SIZE)
        except Exception as e:
            print(f"[WARN] Fehler bei q='{q}', Seite {page}: {e}")
            break

        hits = res.get("hits", [])
        if not hits:
            break

        new_count = 0
        for h in hits:
            tid = h.get("tool", {}).get("id")
            if tid and tid not in seen_ids:
                seen_ids.add(tid)
                all_tools.append(h)
                new_count += 1

        print(f"  Seite {page}: {len(hits)} Treffer, {new_count} neu (gesamt {len(all_tools)})")
        page += 1
        time.sleep(SLEEP_BETWEEN_PAGES)

    # Zwischenspeichern
    with open(SAVE_PATH, "w", encoding="utf-8") as f:
        json.dump(all_tools, f, indent=2, ensure_ascii=False)
    with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
        f.write(q)

    print(f"[INFO] '{q}' abgeschlossen – {len(all_tools)} Tools bisher gespeichert.")


[001/686] Suche nach 'aa'...
  Seite 1: 25 Treffer, 23 neu (gesamt 23)
  Seite 2: 25 Treffer, 21 neu (gesamt 44)
  Seite 3: 25 Treffer, 17 neu (gesamt 61)
  Seite 4: 25 Treffer, 21 neu (gesamt 82)
  Seite 5: 25 Treffer, 25 neu (gesamt 107)
  Seite 6: 25 Treffer, 23 neu (gesamt 130)
  Seite 7: 25 Treffer, 13 neu (gesamt 143)
  Seite 8: 25 Treffer, 22 neu (gesamt 165)
  Seite 9: 25 Treffer, 19 neu (gesamt 184)
  Seite 10: 25 Treffer, 14 neu (gesamt 198)
  Seite 11: 25 Treffer, 24 neu (gesamt 222)
  Seite 12: 25 Treffer, 11 neu (gesamt 233)
  Seite 13: 25 Treffer, 22 neu (gesamt 255)
  Seite 14: 25 Treffer, 24 neu (gesamt 279)
  Seite 15: 25 Treffer, 23 neu (gesamt 302)
  Seite 16: 25 Treffer, 21 neu (gesamt 323)
[WARN] Fehler bei q='aa', Seite 17: GET: error 502: b'<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01 Frameset//EN" "http://www.w3.org/TR/html4/frameset.dtd">\n<html>\n\n<head>\n    <title>Galaxy Tool Shed</title>\n    <link href="/error/style.css" media="screen" rel="stylesheet" t

KeyboardInterrupt: 

In [ ]:
print(f"\n[OK] Abgeschlossen. Gesamt: {len(all_tools)} Tools gespeichert in {SAVE_PATH}")
if os.path.exists(PROGRESS_PATH):
    os.remove(PROGRESS_PATH)
    print("[INFO] Fortschrittsdatei gelöscht – alles fertig.")